# Experiment 2.5: Fine-tuning vs scratch for Muon vs SGD in a deep-linear toy model

**Counterpart script:** `experiments/Tier_1_Core_Mechanism_Tests/FINE_TUNING_muon_vs_sgd/run_experiment.py`

This notebook is an analysis wrapper around the script. It **does not reimplement** the training loop or optimizer logic. Instead, it imports the script, runs the default toy experiment, and presents the returned results in a more explicit academic format.

**Toy-scope question.** After a small target shift, does Muon differ from SGD when:
1. fine-tuning from a shared pre-trained checkpoint, and
2. training from scratch on the modified target?

**Measured here.** Loss trajectories, final losses, steps-to-threshold summaries, and checkpoint **parameter-space** distance during fine-tuning.

**Not measured here.** Lyapunov exponents, direct chaos diagnostics, Hessian geometry, or a gauge-vs-functional decomposition of the motion.


## Notebook workflow

1. Locate and import the counterpart script in a notebook-safe way.
2. Run the default experiment once and log reproducibility/runtime information.
3. Plot the returned fine-tuning and from-scratch trajectories with spread across seeds.
4. Summarize endpoint metrics, threshold-based speed metrics, and the current H1/H2/H3 verdicts.
5. State a calibrated conclusion that distinguishes **early adaptation speed** from **final fine-tune loss**.


In [ ]:
from pathlib import Path
import importlib.util
import platform
import sys
import time

import matplotlib
if 'ipykernel' not in sys.modules:
    matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option('display.max_colwidth', 120)
plt.style.use('seaborn-v0_8-whitegrid')

EXPERIMENT_RELATIVE_DIR = Path('experiments/Tier_1_Core_Mechanism_Tests/FINE_TUNING_muon_vs_sgd')
MODULE_NAME = 'fine_tuning_muon_vs_sgd_experiment'


def locate_experiment_paths(start=None):
    start = Path.cwd().resolve() if start is None else Path(start).resolve()
    search_chain = [start, *start.parents]

    for candidate in search_chain:
        repo_script = candidate / EXPERIMENT_RELATIVE_DIR / 'run_experiment.py'
        if repo_script.exists():
            repo_root = candidate.resolve()
            experiment_dir = (repo_root / EXPERIMENT_RELATIVE_DIR).resolve()
            return repo_root, experiment_dir, repo_script.resolve()

    for candidate in search_chain:
        direct_script = candidate / 'run_experiment.py'
        if candidate.name == 'FINE_TUNING_muon_vs_sgd' and direct_script.exists():
            experiment_dir = candidate.resolve()
            repo_root = None
            for ancestor in [candidate, *candidate.parents]:
                if ancestor.name == 'experiments':
                    repo_root = ancestor.parent.resolve()
                    break
            if repo_root is None:
                repo_root = experiment_dir.parent.resolve()
            return repo_root, experiment_dir, direct_script.resolve()

    raise FileNotFoundError(
        'Could not locate '
        'experiments/Tier_1_Core_Mechanism_Tests/FINE_TUNING_muon_vs_sgd/run_experiment.py '
        'from the current working directory.'
    )


REPO_ROOT, EXPERIMENT_DIR, SCRIPT_PATH = locate_experiment_paths()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

spec = importlib.util.spec_from_file_location(MODULE_NAME, SCRIPT_PATH)
experiment_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(experiment_module)

print(f'Repo root: {REPO_ROOT}')
print(f'Experiment directory: {EXPERIMENT_DIR}')
print(f'Counterpart script: {SCRIPT_PATH}')


## Reproducibility, runtime, and returned result object

The next cell runs the script's `run_experiment()` function with its default configuration and records the runtime seen from the notebook side. The notebook then works entirely from the returned structured result dictionary.


In [ ]:
notebook_t0 = time.perf_counter()
RESULTS = experiment_module.run_experiment(verbose=False)
notebook_runtime_seconds = time.perf_counter() - notebook_t0

CONFIG = RESULTS['config']
RUN_SEEDS = RESULTS['run_seeds']
CURVE_STATS = RESULTS['aggregates']['curve_stats']
FINAL_METRICS = RESULTS['aggregates']['final_metrics']
THRESHOLD_STEPS = RESULTS['aggregates']['threshold_steps']
PAIRED_DIFFS = RESULTS['aggregates']['paired_differences']
EARLY_LATE = RESULTS['aggregates']['early_vs_late_finetune_loss_drop']
HYPOTHESES = RESULTS['hypothesis_tests']

print(f'Notebook cell runtime: {notebook_runtime_seconds:.2f}s')
print(f"Reported experiment runtime: {RESULTS['runtime_seconds']:.2f}s")
print(
    f'Python {platform.python_version()} | numpy {np.__version__} | '
    f'pandas {pd.__version__} | matplotlib {matplotlib.__version__}'
)
print(f'Run seeds: {RUN_SEEDS}')
print(f"Overall conclusion string: {RESULTS['overall_conclusion']}")


## Configuration and scope summary

This cell logs the default setup actually used by the imported script and restates what is and is not being measured.


In [ ]:
config_table = pd.DataFrame([
    {'group': 'model', 'parameter': 'dim', 'value': CONFIG['dim']},
    {'group': 'model', 'parameter': 'num_layers', 'value': CONFIG['num_layers']},
    {'group': 'data', 'parameter': 'batch_size', 'value': CONFIG['batch_size']},
    {'group': 'data', 'parameter': 'modify_frac', 'value': CONFIG['modify_frac']},
    {'group': 'pretrain', 'parameter': 'steps', 'value': CONFIG['pretrain_steps']},
    {'group': 'pretrain', 'parameter': 'lr', 'value': CONFIG['pretrain_lr']},
    {'group': 'fine_tune', 'parameter': 'steps', 'value': CONFIG['finetune_steps']},
    {'group': 'fine_tune', 'parameter': 'sgd_lr', 'value': CONFIG['sgd_finetune_lr']},
    {'group': 'fine_tune', 'parameter': 'muon_lr', 'value': CONFIG['muon_finetune_lr']},
    {'group': 'scratch', 'parameter': 'steps', 'value': CONFIG['scratch_steps']},
    {'group': 'scratch', 'parameter': 'sgd_lr', 'value': CONFIG['sgd_scratch_lr']},
    {'group': 'scratch', 'parameter': 'muon_lr', 'value': CONFIG['muon_scratch_lr']},
    {'group': 'muon', 'parameter': 'momentum', 'value': CONFIG['momentum']},
    {'group': 'muon', 'parameter': 'ns_iters', 'value': CONFIG['ns_iters']},
    {'group': 'reproducibility', 'parameter': 'seed', 'value': CONFIG['seed']},
    {'group': 'reproducibility', 'parameter': 'seed_stride', 'value': CONFIG['seed_stride']},
    {'group': 'reproducibility', 'parameter': 'num_runs', 'value': CONFIG['num_runs']},
])

measurement_table = pd.DataFrame([
    {'category': 'Measured', 'item': 'Pre-training loss curves'},
    {'category': 'Measured', 'item': 'Fine-tuning loss curves'},
    {'category': 'Measured', 'item': 'From-scratch loss curves'},
    {'category': 'Measured', 'item': 'Fine-tuning checkpoint parameter distance'},
    {'category': 'Measured', 'item': 'Steps to 50% of initial loss'},
    {'category': 'Not measured', 'item': 'Lyapunov exponents'},
    {'category': 'Not measured', 'item': 'Direct chaos diagnostic'},
    {'category': 'Not measured', 'item': 'Hessian curvature'},
    {'category': 'Not measured', 'item': 'Gauge-vs-functional decomposition'},
])

print('Configuration')
print(config_table.to_string(index=False))
print()
print('Base problem summary')
print(pd.DataFrame([RESULTS['base_problem_summary']]).to_string(index=False))
print()
print('Measurement and scope summary')
print(measurement_table.to_string(index=False))


In [ ]:
def plot_mean_with_spread(ax, stats_key, label, color, ylabel):
    stats = CURVE_STATS[stats_key]
    mean_curve = stats['mean']
    sd_curve = stats['sd']
    all_curves = stats['all_curves']
    x = np.arange(mean_curve.size)

    for curve in all_curves:
        ax.plot(x, curve, color=color, alpha=0.15, linewidth=1.0)
    ax.plot(x, mean_curve, color=color, linewidth=2.5, label=f'{label} mean')
    ax.fill_between(
        x,
        mean_curve - sd_curve,
        mean_curve + sd_curve,
        color=color,
        alpha=0.22,
        label=f'{label} ± 1 sample sd',
    )
    ax.set_xlabel('Step')
    ax.set_ylabel(ylabel)
    ax.grid(alpha=0.3)


def mean_sd_text(summary, precision=4):
    return f"{summary['mean']:.{precision}f} ± {summary['sd']:.{precision}f}"


## Fine-tuning loss trajectories

The plot below shows fine-tuning loss from the shared checkpoint to the modified target. Faint traces are individual runs; bold curves show the across-run mean; shaded bands show ±1 sample standard deviation.


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
plot_mean_with_spread(ax, 'ft_sgd_losses', 'SGD', 'tab:blue', 'Loss')
plot_mean_with_spread(ax, 'ft_muon_losses', 'Muon', 'tab:orange', 'Loss')
ax.set_title('Fine-tuning from checkpoint: loss vs step')
ax.legend(frameon=True)
plt.tight_layout()
plt.show()


In [ ]:
print('Fine-tuning interpretation')
print(
    f'- Half-loss speed: SGD reaches 50% of its initial fine-tune loss in '
    f"{THRESHOLD_STEPS['ft_sgd_half_loss']['mean']:.1f} ± {THRESHOLD_STEPS['ft_sgd_half_loss']['sd']:.1f} steps, "
    f"whereas Muon takes {THRESHOLD_STEPS['ft_muon_half_loss']['mean']:.1f} ± {THRESHOLD_STEPS['ft_muon_half_loss']['sd']:.1f} steps."
)
print(
    f'- Endpoint loss: SGD finishes at {mean_sd_text(FINAL_METRICS["ft_sgd_final_loss"], 6)}, '
    f'whereas Muon finishes at {mean_sd_text(FINAL_METRICS["ft_muon_final_loss"], 6)}.'
)
print(
    '- Therefore the current run separates early-speed and final-loss conclusions: '
    'SGD improves faster early, while Muon reaches the lower final fine-tune loss at the 200-step budget.'
)


## Fine-tuning checkpoint-distance trajectories

This is a **parameter-space** distance from the saved checkpoint, aggregated over all layers. It is useful as a movement proxy, but it is **not** a direct measure of gauge-only motion and does not by itself establish chaotic behavior.


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
plot_mean_with_spread(ax, 'ft_sgd_dists', 'SGD', 'tab:blue', 'Checkpoint parameter distance')
plot_mean_with_spread(ax, 'ft_muon_dists', 'Muon', 'tab:orange', 'Checkpoint parameter distance')
ax.set_title('Fine-tuning from checkpoint: parameter distance vs step')
ax.legend(frameon=True)
plt.tight_layout()
plt.show()


In [ ]:
print('Checkpoint-distance interpretation')
print(
    f'- Final checkpoint distance: SGD = {mean_sd_text(FINAL_METRICS["ft_sgd_final_checkpoint_distance"], 4)}, '
    f'Muon = {mean_sd_text(FINAL_METRICS["ft_muon_final_checkpoint_distance"], 4)}.'
)
print('- In the current run, Muon ends farther from the checkpoint by the end of fine-tuning.')
print(
    '- This supports an endpoint movement difference in parameter space, but it should not be over-read '
    'as a direct gauge-direction measurement or as proof of chaos.'
)


## From-scratch reference trajectories

The experiment also includes a matched from-scratch comparison on the modified target. This provides context for the optimizer behavior outside the checkpoint-adaptation setting.


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
plot_mean_with_spread(ax, 'scratch_sgd_losses', 'SGD', 'tab:blue', 'Loss')
plot_mean_with_spread(ax, 'scratch_muon_losses', 'Muon', 'tab:orange', 'Loss')
ax.set_title('From scratch on modified target: loss vs step')
ax.legend(frameon=True)
plt.tight_layout()
plt.show()


In [ ]:
print('From-scratch interpretation')
print(
    f'- Final scratch loss: SGD = {mean_sd_text(FINAL_METRICS["scratch_sgd_final_loss"], 6)}, '
    f'Muon = {mean_sd_text(FINAL_METRICS["scratch_muon_final_loss"], 6)}.'
)
print(
    f'- Half-loss speed from scratch: SGD = {THRESHOLD_STEPS["scratch_sgd_half_loss"]["mean"]:.1f} ± '
    f"{THRESHOLD_STEPS['scratch_sgd_half_loss']['sd']:.1f} steps, Muon = "
    f"{THRESHOLD_STEPS['scratch_muon_half_loss']['mean']:.1f} ± {THRESHOLD_STEPS['scratch_muon_half_loss']['sd']:.1f} steps."
)
print(
    '- As in fine-tuning, SGD is faster early by this threshold metric, but Muon wins on the final 700-step endpoint.'
)


## Summary tables

The following tables collect the endpoint losses, checkpoint-distance proxy, threshold-based speed summaries, paired seedwise differences, and early-vs-late fine-tuning loss-drop summaries.


In [ ]:
final_table = pd.DataFrame([
    {
        'metric': 'Fine-tune final loss',
        'SGD mean': FINAL_METRICS['ft_sgd_final_loss']['mean'],
        'SGD sd': FINAL_METRICS['ft_sgd_final_loss']['sd'],
        'Muon mean': FINAL_METRICS['ft_muon_final_loss']['mean'],
        'Muon sd': FINAL_METRICS['ft_muon_final_loss']['sd'],
    },
    {
        'metric': 'Fine-tune final checkpoint distance',
        'SGD mean': FINAL_METRICS['ft_sgd_final_checkpoint_distance']['mean'],
        'SGD sd': FINAL_METRICS['ft_sgd_final_checkpoint_distance']['sd'],
        'Muon mean': FINAL_METRICS['ft_muon_final_checkpoint_distance']['mean'],
        'Muon sd': FINAL_METRICS['ft_muon_final_checkpoint_distance']['sd'],
    },
    {
        'metric': 'Scratch final loss',
        'SGD mean': FINAL_METRICS['scratch_sgd_final_loss']['mean'],
        'SGD sd': FINAL_METRICS['scratch_sgd_final_loss']['sd'],
        'Muon mean': FINAL_METRICS['scratch_muon_final_loss']['mean'],
        'Muon sd': FINAL_METRICS['scratch_muon_final_loss']['sd'],
    },
])

threshold_table = pd.DataFrame([
    {
        'scenario': 'Fine-tune: steps to 50% initial loss',
        'SGD mean': THRESHOLD_STEPS['ft_sgd_half_loss']['mean'],
        'SGD sd': THRESHOLD_STEPS['ft_sgd_half_loss']['sd'],
        'Muon mean': THRESHOLD_STEPS['ft_muon_half_loss']['mean'],
        'Muon sd': THRESHOLD_STEPS['ft_muon_half_loss']['sd'],
    },
    {
        'scenario': 'Scratch: steps to 50% initial loss',
        'SGD mean': THRESHOLD_STEPS['scratch_sgd_half_loss']['mean'],
        'SGD sd': THRESHOLD_STEPS['scratch_sgd_half_loss']['sd'],
        'Muon mean': THRESHOLD_STEPS['scratch_muon_half_loss']['mean'],
        'Muon sd': THRESHOLD_STEPS['scratch_muon_half_loss']['sd'],
    },
])

paired_diff_table = pd.DataFrame([
    {
        'paired quantity (Muon - SGD)': 'Fine-tune final loss',
        'mean': PAIRED_DIFFS['finetune_final_loss_muon_minus_sgd']['mean'],
        'sd': PAIRED_DIFFS['finetune_final_loss_muon_minus_sgd']['sd'],
    },
    {
        'paired quantity (Muon - SGD)': 'Fine-tune final checkpoint distance',
        'mean': PAIRED_DIFFS['finetune_final_checkpoint_distance_muon_minus_sgd']['mean'],
        'sd': PAIRED_DIFFS['finetune_final_checkpoint_distance_muon_minus_sgd']['sd'],
    },
    {
        'paired quantity (Muon - SGD)': 'Scratch final loss',
        'mean': PAIRED_DIFFS['scratch_final_loss_muon_minus_sgd']['mean'],
        'sd': PAIRED_DIFFS['scratch_final_loss_muon_minus_sgd']['sd'],
    },
])

early_late_table = pd.DataFrame([
    {
        'optimizer': 'SGD',
        'early drop 0->50': EARLY_LATE['sgd']['early_drop_0_to_50']['mean'],
        'early drop sd': EARLY_LATE['sgd']['early_drop_0_to_50']['sd'],
        'late drop 150->200': EARLY_LATE['sgd']['late_drop_150_to_200']['mean'],
        'late drop sd': EARLY_LATE['sgd']['late_drop_150_to_200']['sd'],
    },
    {
        'optimizer': 'Muon',
        'early drop 0->50': EARLY_LATE['muon']['early_drop_0_to_50']['mean'],
        'early drop sd': EARLY_LATE['muon']['early_drop_0_to_50']['sd'],
        'late drop 150->200': EARLY_LATE['muon']['late_drop_150_to_200']['mean'],
        'late drop sd': EARLY_LATE['muon']['late_drop_150_to_200']['sd'],
    },
])

pretrain_table = pd.DataFrame([
    {
        'mean loss': FINAL_METRICS['pretrain_final_loss']['mean'],
        'sample sd': FINAL_METRICS['pretrain_final_loss']['sd'],
        'n': FINAL_METRICS['pretrain_final_loss']['n'],
    }
])

print('Pre-training final loss (SGD only)')
print(pretrain_table.round(6).to_string(index=False))
print()
print('Final metrics')
print(final_table.round(6).to_string(index=False))
print()
print('Half-threshold summaries')
print(threshold_table.round(3).to_string(index=False))
print()
print('Paired seedwise differences')
print(paired_diff_table.round(6).to_string(index=False))
print()
print('Fine-tuning early vs late loss-drop summary')
print(early_late_table.round(6).to_string(index=False))


In [ ]:
print('Summary-table interpretation')
print("- The current result is not well described by a single 'better for fine-tuning' scalar claim.")
print(
    f"- Early-speed proxy: fine-tune half-loss steps favor SGD ({THRESHOLD_STEPS['ft_sgd_half_loss']['mean']:.1f}) "
    f"over Muon ({THRESHOLD_STEPS['ft_muon_half_loss']['mean']:.1f})."
)
print(
    f"- Final fine-tune endpoint: final loss favors Muon ({FINAL_METRICS['ft_muon_final_loss']['mean']:.6f}) "
    f"over SGD ({FINAL_METRICS['ft_sgd_final_loss']['mean']:.6f})."
)
print("- Thus, in this toy setup, 'slower early adaptation' and 'worse final fine-tune loss' come apart.")


## H1/H2/H3 verdicts for the current default run

The script now returns explicit question text, measured quantities, and verdicts for the three original high-level checks.


In [ ]:
hypothesis_rows = []
for key in ['H1', 'H2', 'H3']:
    item = HYPOTHESES[key]
    hypothesis_rows.append({
        'hypothesis': key,
        'question': item['question'],
        'measured quantity': item['measured_quantity'],
        'Muon mean': item['lhs_mean'],
        'SGD mean': item['rhs_mean'],
        'verdict': item['verdict'],
    })

hypothesis_table = pd.DataFrame(hypothesis_rows)
print(hypothesis_table.round(6).to_string(index=False))


## Calibrated conclusion

The notebook conclusion should match the computed results rather than the original over-strong narrative.


In [ ]:
print('Calibrated conclusion')
print(RESULTS['overall_conclusion'])
print()
print('Limitations carried through from the script')
for limitation in RESULTS['limitations']:
    print(f'- {limitation}')
